# Signal-noise ratio

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, Video, display
from pathlib import Path
import sys

# Locate the shared Telemetry visualization library from anywhere inside the monorepo.
TELEMETRY_ROOT = next(
    (
        root / "products/visualization-studio/visualization-studio-content"
        for root in (Path.cwd(), *Path.cwd().parents)
        if (root / "products/visualization-studio/visualization-studio-content/vizlib/animation_export.py").exists()
    ),
    None,
)
if TELEMETRY_ROOT is None:
    raise FileNotFoundError("Could not locate Telemetry vizlib in the IncusLuminis monorepo")

if str(TELEMETRY_ROOT) not in sys.path:
    sys.path.insert(0, str(TELEMETRY_ROOT))

from vizlib.animation_export import export_animation, normalize_output_format

# =========================
# Config
# =========================

OUT_DIR = Path("animations")
OUT_DIR.mkdir(parents=True, exist_ok=True)

ANIMATION_NAME = "exoplanet_signal_vs_star_noise"
OUTPUT_FORMAT = normalize_output_format("mp4")  # webm | gif | mp4


FPS = 24
DURATION_SEC = 12
FRAMES = FPS * DURATION_SEC

FIGSIZE = (16, 9)
DPI = 120

BG = "#030711"
GRID = "#16324a"
TEXT = "#c9d3df"
MUTED = "#6f8194"

STAR = "#ffd36a"
STAR_FILL = "#ffb347"
PLANET = "#35c9ff"
PLANET_FILL = "#0b63ff"
NOISE = "#ff4dff"
WHITE = "#f0f6ff"
ORANGE = "#ff6b32"
GREEN = "#48ffb3"

rng = np.random.default_rng(32)

# =========================
# Synthetic signal model
# =========================

x = np.linspace(0, 1, 1400)

def gaussian(x, mu, amp, sigma):
    return amp * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

# Star: huge broad signal
star_signal = gaussian(x, 0.48, 1.0, 0.105)

# Planet: tiny narrow signal, close to the star PSF wing
planet_signal = gaussian(x, 0.66, 0.020, 0.018)

# Structured stellar residual + detector noise
speckle_noise = (
    0.010 * np.sin(2 * np.pi * 19 * x + 0.6)
    + 0.007 * np.sin(2 * np.pi * 43 * x + 1.4)
    + 0.004 * np.sin(2 * np.pi * 81 * x + 0.2)
)

random_noise = rng.normal(0, 0.0045, size=x.size)
noise = speckle_noise + random_noise

observed = star_signal + planet_signal + noise

# Stellar model subtraction: imperfect PSF removal
star_model = gaussian(x, 0.48, 0.995, 0.107)
residual = observed - star_model

# For plotting residual on comparable lower axis
residual_scaled = residual

planet_only = planet_signal

# =========================
# Figure setup
# =========================

fig, (ax, axr) = plt.subplots(
    2, 1,
    figsize=FIGSIZE,
    dpi=DPI,
    sharex=True,
    gridspec_kw={"height_ratios": [3.2, 1.2], "hspace": 0.08}
)

fig.patch.set_facecolor(BG)

for a in (ax, axr):
    a.set_facecolor(BG)
    a.grid(True, color=GRID, linestyle=":", linewidth=1.0, alpha=0.75)
    a.tick_params(colors=TEXT, labelsize=12, length=5)

    for spine in a.spines.values():
        spine.set_color("#95a3b5")
        spine.set_linewidth(1.1)

ax.set_xlim(0, 1)
ax.set_ylim(-0.04, 1.12)
axr.set_ylim(-0.045, 0.055)

ax.set_ylabel("Relative intensity", color=TEXT, fontsize=17, labelpad=12)
axr.set_ylabel("Residual", color=TEXT, fontsize=15, labelpad=12)
axr.set_xlabel("Angular separation / detector coordinate", color=TEXT, fontsize=17, labelpad=12)

ax.set_xticks(np.linspace(0, 1, 6))
ax.set_xticklabels(["0", "0.2", "0.4", "0.6", "0.8", "1.0"])

fig.text(
    0.5, 0.945,
    "EXOPLANET SIGNAL VS STELLAR SIGNAL",
    ha="center",
    va="center",
    color="#d7dde8",
    fontsize=24,
    fontweight="bold"
)

fig.text(
    0.5, 0.905,
    "the planet is not absent — it is buried under the star's photon noise and residual starlight",
    ha="center",
    va="center",
    color=MUTED,
    fontsize=13,
    fontweight="bold"
)

# =========================
# Static guide lines and labels
# =========================

star_x = 0.48
planet_x = 0.66

ax.axvline(star_x, color=STAR, linestyle=":", linewidth=1.1, alpha=0.35)
ax.axvline(planet_x, color=PLANET, linestyle=":", linewidth=1.1, alpha=0.45)

axr.axvline(planet_x, color=PLANET, linestyle=":", linewidth=1.0, alpha=0.35)
axr.axhline(0, color=TEXT, linewidth=0.9, alpha=0.35)

ax.text(
    star_x,
    1.06,
    "STAR",
    color=STAR,
    fontsize=15,
    fontweight="bold",
    ha="center"
)

ax.text(
    planet_x,
    0.105,
    "PLANET\n~1/50 OF STAR PEAK",
    color=PLANET,
    fontsize=12,
    fontweight="bold",
    ha="center",
    va="bottom"
)

ax.text(
    0.04,
    0.98,
    "raw observation: star + planet + noise",
    transform=ax.transAxes,
    color=TEXT,
    fontsize=13,
    fontweight="bold",
    ha="left",
    va="top"
)

axr.text(
    0.04,
    0.90,
    "after stellar model subtraction: planet remains near the noise floor",
    transform=axr.transAxes,
    color=TEXT,
    fontsize=12,
    fontweight="bold",
    ha="left",
    va="top"
)

# Ratio box
ratio_text = (
    "illustrative contrast\n"
    "star peak  = 1.000\n"
    "planet peak = 0.020\n"
    "contrast ≈ 50:1\n\n"
    "real direct imaging\n"
    "often requires\n"
    "10⁶–10¹⁰ contrast"
)

ax.text(
    0.96,
    0.93,
    ratio_text,
    transform=ax.transAxes,
    color=ORANGE,
    fontsize=11,
    ha="right",
    va="top",
    bbox=dict(
        boxstyle="round,pad=0.45",
        facecolor="#07111f",
        edgecolor=ORANGE,
        alpha=0.86
    ),
    zorder=30
)

# =========================
# Animated artists
# =========================

star_fill = ax.fill_between([], [], [], color=STAR_FILL, alpha=0.0)
planet_fill = ax.fill_between([], [], [], color=PLANET_FILL, alpha=0.0)

star_glow1, = ax.plot([], [], color=STAR, linewidth=10, alpha=0.10, zorder=12)
star_glow2, = ax.plot([], [], color=STAR, linewidth=5, alpha=0.18, zorder=13)
star_line, = ax.plot([], [], color=STAR, linewidth=2.5, zorder=14)

obs_line, = ax.plot([], [], color=WHITE, linewidth=1.0, alpha=0.75, zorder=16)

planet_glow, = ax.plot([], [], color=PLANET, linewidth=8, alpha=0.0, zorder=20)
planet_line, = ax.plot([], [], color=PLANET, linewidth=2.4, alpha=0.0, zorder=21)

noise_line, = ax.plot([], [], color=NOISE, linewidth=1.0, alpha=0.0, zorder=15)

res_glow, = axr.plot([], [], color=PLANET, linewidth=7, alpha=0.0, zorder=12)
res_line, = axr.plot([], [], color=PLANET, linewidth=1.8, alpha=0.0, zorder=13)
res_noise_line, = axr.plot([], [], color=NOISE, linewidth=1.0, alpha=0.0, zorder=11)

planet_marker = ax.scatter([], [], s=180, color=PLANET, edgecolor=WHITE, linewidths=1.0, alpha=0.0, zorder=25)
res_marker = axr.scatter([], [], s=120, color=PLANET, edgecolor=WHITE, linewidths=1.0, alpha=0.0, zorder=25)

status = ax.text(
    0.04,
    0.06,
    "stage: raw stellar glare",
    transform=ax.transAxes,
    color=MUTED,
    fontsize=12,
    ha="left",
    va="bottom",
    bbox=dict(
        boxstyle="round,pad=0.35",
        facecolor="#07111f",
        edgecolor="#2b4358",
        alpha=0.85
    ),
    zorder=30
)

# =========================
# Animation helpers
# =========================

def smoothstep(edge0, edge1, value):
    t = np.clip((value - edge0) / (edge1 - edge0), 0, 1)
    return t * t * (3 - 2 * t)

def ease(t):
    return 1 - (1 - t) ** 3

def draw_alpha(t):
    fade_in = smoothstep(0.03, 0.12, t)
    fade_out = 1.0 - smoothstep(0.92, 0.99, t)
    return fade_in * fade_out

# =========================
# Animation update
# =========================

def update(frame):
    global star_fill, planet_fill

    t = frame / (FRAMES - 1)
    a = draw_alpha(t)

    scan = ease(t)
    xmax = x.min() + scan * (x.max() - x.min())
    visible = x <= xmax

    xv = x[visible]

    # Stage alphas
    star_a = a
    obs_a = a * smoothstep(0.14, 0.26, t)
    noise_a = a * smoothstep(0.20, 0.34, t)
    planet_a = a * smoothstep(0.38, 0.55, t)
    residual_a = a * smoothstep(0.58, 0.76, t)

    # Upper panel lines
    star_line.set_data(xv, star_signal[visible])
    star_glow1.set_data(xv, star_signal[visible])
    star_glow2.set_data(xv, star_signal[visible])

    obs_line.set_data(xv, observed[visible])
    obs_line.set_alpha(0.55 * obs_a)

    noise_line.set_data(xv, star_signal[visible] + noise[visible])
    noise_line.set_alpha(0.55 * noise_a)

    planet_line.set_data(xv, planet_signal[visible])
    planet_line.set_alpha(0.95 * planet_a)

    planet_glow.set_data(xv, planet_signal[visible])
    planet_glow.set_alpha(0.18 * planet_a)

    # Fills
    star_fill.remove()
    planet_fill.remove()

    star_fill = ax.fill_between(
        xv,
        0,
        star_signal[visible],
        color=STAR_FILL,
        alpha=0.18 * star_a
    )

    planet_fill = ax.fill_between(
        xv,
        0,
        planet_signal[visible],
        color=PLANET_FILL,
        alpha=0.40 * planet_a
    )

    # Residual panel
    res_noise_line.set_data(xv, residual_scaled[visible])
    res_noise_line.set_alpha(0.55 * residual_a)

    res_line.set_data(xv, planet_only[visible])
    res_line.set_alpha(0.95 * residual_a)

    res_glow.set_data(xv, planet_only[visible])
    res_glow.set_alpha(0.14 * residual_a)

    # Markers after scan reaches planet
    planet_passed = smoothstep(planet_x - 0.025, planet_x + 0.035, xmax)
    pulse = 0.5 + 0.5 * np.sin(2 * np.pi * 6 * t)

    py = np.interp(planet_x, x, planet_signal)
    ry = np.interp(planet_x, x, residual_scaled)

    planet_marker.set_offsets([[planet_x, py]])
    res_marker.set_offsets([[planet_x, ry]])

    planet_marker.set_alpha(planet_a * planet_passed * (0.65 + 0.35 * pulse))
    res_marker.set_alpha(residual_a * planet_passed * (0.65 + 0.35 * pulse))

    planet_marker.set_sizes([150 + 90 * pulse])
    res_marker.set_sizes([100 + 60 * pulse])

    # Status text
    if t < 0.22:
        status.set_text("stage: raw stellar glare")
        status.set_color(STAR)
    elif t < 0.42:
        status.set_text("stage: photon noise and speckles dominate")
        status.set_color(NOISE)
    elif t < 0.62:
        status.set_text("stage: tiny planet signal is indicated")
        status.set_color(PLANET)
    else:
        status.set_text("stage: subtract star model → search residuals")
        status.set_color(GREEN)

    return (
        [star_line, star_glow1, star_glow2,
         obs_line, noise_line,
         planet_line, planet_glow,
         res_line, res_glow, res_noise_line,
         planet_marker, res_marker,
         status, star_fill, planet_fill]
    )

# =========================
# Render and export
# =========================

frames = []
for frame_idx in range(FRAMES):
    update(frame_idx)
    fig.canvas.draw()
    rgba = np.asarray(fig.canvas.buffer_rgba(), dtype=np.uint8)
    frames.append(rgba[..., :3].copy())

plt.close(fig)

out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
)

if OUTPUT_FORMAT == "gif":
    display(Image(filename=str(out_file)))
else:
    display(Video(filename=str(out_file), embed=False))

print(f"Saved: {out_file}")


ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/exoplanet_signal_vs_star_noise.mp4


In [4]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, Video, display
from pathlib import Path
import sys

# Locate the shared Telemetry visualization library from anywhere inside the monorepo.
TELEMETRY_ROOT = next(
    (
        root / "products/visualization-studio/visualization-studio-content"
        for root in (Path.cwd(), *Path.cwd().parents)
        if (root / "products/visualization-studio/visualization-studio-content/vizlib/animation_export.py").exists()
    ),
    None,
)
if TELEMETRY_ROOT is None:
    raise FileNotFoundError("Could not locate Telemetry vizlib in the IncusLuminis monorepo")

if str(TELEMETRY_ROOT) not in sys.path:
    sys.path.insert(0, str(TELEMETRY_ROOT))

from vizlib.animation_export import export_animation, normalize_output_format

# =========================
# Config
# =========================

OUT_DIR = Path("animations/")
OUT_DIR.mkdir(parents=True, exist_ok=True)

ANIMATION_NAME = "exoplanet_signal_in_white_noise"
OUTPUT_FORMAT = normalize_output_format("webm")  # webm | gif | mp4


FPS = 24
DURATION_SEC = 12
FRAMES = FPS * DURATION_SEC

FIGSIZE = (16, 9)
DPI = 120

BG = "#030711"
GRID = "#16324a"
TEXT = "#c9d3df"
MUTED = "#6f8194"

NOISE = "#ff4dff"
PLANET = "#35c9ff"
PLANET_FILL = "#0b63ff"
WHITE = "#f0f6ff"
GREEN = "#48ffb3"
ORANGE = "#ff6b32"

rng = np.random.default_rng(44)

# =========================
# Signal model
# =========================

x = np.linspace(0, 1, 900)

planet_x = 0.64
planet_amp = 0.008
planet_sigma = 0.008
noise_sigma = 0.024

def gaussian(x, mu, amp, sigma):
    return amp * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

planet_signal = gaussian(x, planet_x, planet_amp, planet_sigma)

# fixed envelope for reference only
detection_window = (x > planet_x - 0.06) & (x < planet_x + 0.06)

# =========================
# Figure setup
# =========================

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

fig.subplots_adjust(left=0.08, right=0.98, top=0.88, bottom=0.14)

ax.set_xlim(0, 1)
ax.set_ylim(-0.055, 0.065)

ax.grid(True, color=GRID, linestyle=":", linewidth=1.0, alpha=0.75)

for spine in ax.spines.values():
    spine.set_color("#95a3b5")
    spine.set_linewidth(1.1)

ax.tick_params(colors=TEXT, labelsize=13, length=5)

ax.set_xlabel("Angular separation / detector coordinate", color=TEXT, fontsize=18, labelpad=14)
ax.set_ylabel("Residual intensity", color=TEXT, fontsize=18, labelpad=14)

fig.text(
    0.5, 0.94,
    "EXOPLANET SIGNAL BURIED IN WHITE NOISE",
    ha="center",
    va="center",
    color="#d7dde8",
    fontsize=24,
    fontweight="bold"
)

fig.text(
    0.5, 0.90,
    "the planet is a weak fixed signal; the noise realization changes constantly",
    ha="center",
    va="center",
    color=MUTED,
    fontsize=13,
    fontweight="bold"
)

# Static guides
ax.axhline(0, color=TEXT, linewidth=1.0, alpha=0.35)
ax.axvline(planet_x, color=PLANET, linestyle=":", linewidth=1.2, alpha=0.45)

ax.axvspan(
    planet_x - 0.045,
    planet_x + 0.045,
    color=PLANET_FILL,
    alpha=0.08,
    zorder=0
)

ax.text(
    planet_x,
    0.059,
    "expected planet position",
    color=PLANET,
    fontsize=12,
    fontweight="bold",
    ha="center",
    va="top"
)

# SNR box
snr_text = (
    "illustrative residual frame\n"
    f"planet amplitude ≈ {planet_amp:.3f}\n"
    f"noise σ ≈ {noise_sigma:.3f}\n"
    f"single-frame S/N ≈ {planet_amp / noise_sigma:.1f}"
)

ax.text(
    0.97,
    0.92,
    snr_text,
    transform=ax.transAxes,
    color=ORANGE,
    fontsize=12,
    ha="right",
    va="top",
    bbox=dict(
        boxstyle="round,pad=0.45",
        facecolor="#07111f",
        edgecolor=ORANGE,
        alpha=0.86
    ),
    zorder=30
)

# =========================
# Animated artists
# =========================

noise_line, = ax.plot([], [], color=NOISE, linewidth=1.1, alpha=0.65, zorder=10)
noise_glow, = ax.plot([], [], color=NOISE, linewidth=5, alpha=0.08, zorder=9)

planet_line, = ax.plot([], [], color=PLANET, linewidth=2.4, alpha=0.95, zorder=15)
planet_glow, = ax.plot([], [], color=PLANET, linewidth=8, alpha=0.14, zorder=14)

combined_line, = ax.plot([], [], color=WHITE, linewidth=1.0, alpha=0.65, zorder=12)

planet_marker = ax.scatter(
    [], [],
    s=160,
    color=PLANET,
    edgecolor=WHITE,
    linewidths=1.0,
    alpha=0.0,
    zorder=20
)

window_line, = ax.plot(
    [],
    [],
    color=GREEN,
    linewidth=2.0,
    alpha=0.0,
    zorder=18
)

status = ax.text(
    0.04,
    0.06,
    "white noise frame: signal hidden",
    transform=ax.transAxes,
    color=MUTED,
    fontsize=12,
    ha="left",
    va="bottom",
    bbox=dict(
        boxstyle="round,pad=0.35",
        facecolor="#07111f",
        edgecolor="#2b4358",
        alpha=0.85
    ),
    zorder=30
)

# =========================
# Animation helpers
# =========================

def smoothstep(edge0, edge1, value):
    t = np.clip((value - edge0) / (edge1 - edge0), 0, 1)
    return t * t * (3 - 2 * t)

def layer_alpha(t):
    fade_in = smoothstep(0.04, 0.14, t)
    fade_out = 1.0 - smoothstep(0.92, 0.99, t)
    return fade_in * fade_out

def correlated_white_noise(rng, n, sigma):
    """
    Mostly white noise, slightly smoothed so the curve remains readable as a plot.
    """
    raw = rng.normal(0, sigma, size=n)
    kernel = np.array([0.18, 0.64, 0.18])
    return np.convolve(raw, kernel, mode="same")

# =========================
# Animation update
# =========================

def update(frame):
    t = frame / (FRAMES - 1)
    a = layer_alpha(t)

    # New noise realization every frame
    current_noise = correlated_white_noise(rng, len(x), noise_sigma)
    observed_residual = current_noise + planet_signal

    # Visual pulse around the known planet position
    pulse = 0.5 + 0.5 * np.sin(2 * np.pi * 5 * t)

    # Noise and combined residual
    noise_line.set_data(x, current_noise)
    noise_line.set_alpha(0.55 * a)

    noise_glow.set_data(x, current_noise)
    noise_glow.set_alpha(0.06 * a)

    combined_line.set_data(x, observed_residual)
    combined_line.set_alpha(0.80 * a)

    # Underlying planet signal model
    reveal = smoothstep(0.28, 0.48, t) * (1.0 - smoothstep(0.82, 0.95, t))
    planet_a = a * (0.18 + 0.82 * reveal)

    planet_line.set_data(x, planet_signal)
    planet_line.set_alpha(0.95 * planet_a)

    planet_glow.set_data(x, planet_signal)
    planet_glow.set_alpha((0.10 + 0.12 * pulse) * planet_a)

    # Detection window highlight
    window_x = x[detection_window]
    window_y = observed_residual[detection_window]
    window_line.set_data(window_x, window_y)
    window_line.set_alpha(0.75 * reveal * a)

    # Marker at current noisy value at planet position
    py_observed = np.interp(planet_x, x, observed_residual)
    planet_marker.set_offsets([[planet_x, py_observed]])
    planet_marker.set_alpha((0.35 + 0.65 * reveal) * a)
    planet_marker.set_sizes([120 + 110 * pulse])

    if reveal < 0.25:
        status.set_text("white noise frame: signal hidden")
        status.set_color(MUTED)
    elif reveal < 0.85:
        status.set_text("known position: weak excess begins to emerge")
        status.set_color(PLANET)
    else:
        status.set_text("planet signal: fixed feature inside changing noise")
        status.set_color(GREEN)

    return (
        noise_line,
        noise_glow,
        combined_line,
        planet_line,
        planet_glow,
        window_line,
        planet_marker,
        status,
    )

# =========================
# Render and export
# =========================

frames = []
for frame_idx in range(FRAMES):
    update(frame_idx)
    fig.canvas.draw()
    rgba = np.asarray(fig.canvas.buffer_rgba(), dtype=np.uint8)
    frames.append(rgba[..., :3].copy())

plt.close(fig)

out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
)

if OUTPUT_FORMAT == "gif":
    display(Image(filename=str(out_file)))
else:
    display(Video(filename=str(out_file), embed=False))

print(f"Saved: {out_file}")


ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/exoplanet_signal_in_white_noise.webm


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from IPython.display import Image, Video, display
from pathlib import Path
import sys

# Locate the shared Telemetry visualization library from anywhere inside the monorepo.
TELEMETRY_ROOT = next(
    (
        root / "products/visualization-studio/visualization-studio-content"
        for root in (Path.cwd(), *Path.cwd().parents)
        if (root / "products/visualization-studio/visualization-studio-content/vizlib/animation_export.py").exists()
    ),
    None,
)
if TELEMETRY_ROOT is None:
    raise FileNotFoundError("Could not locate Telemetry vizlib in the IncusLuminis monorepo")

if str(TELEMETRY_ROOT) not in sys.path:
    sys.path.insert(0, str(TELEMETRY_ROOT))

from vizlib.animation_export import export_animation, normalize_output_format

# =========================
# Config
# =========================

OUT_DIR = Path("animations/")
OUT_DIR.mkdir(parents=True, exist_ok=True)

ANIMATION_NAME = "exoplanet_2d_coronagraph_physical_noise"
OUTPUT_FORMAT = normalize_output_format("webm")  # webm | gif | mp4


FPS = 24
DURATION_SEC = 12
FRAMES = FPS * DURATION_SEC

FIGSIZE = (16, 9)
DPI = 120

BG = "#030711"
TEXT = "#c9d3df"
MUTED = "#6f8194"

PLANET = "#35c9ff"
WHITE = "#f0f6ff"
ORANGE = "#ff6b32"
GREEN = "#48ffb3"

rng = np.random.default_rng(71)

# =========================
# Detector geometry
# =========================

N = 240
yy, xx = np.mgrid[-1:1:complex(N), -1:1:complex(N)]
rr = np.sqrt(xx**2 + yy**2)
theta = np.arctan2(yy, xx)

planet_x, planet_y = 0.43, -0.22

coronagraph_radius = 0.17

# =========================
# Physical-ish signal model
# =========================

def gaussian2d(x, y, x0, y0, amp, sigma):
    return amp * np.exp(
        -0.5 * (((x - x0) ** 2 + (y - y0) ** 2) / sigma**2)
    )

# Planet: faint, diffraction-limited point source
planet_amp = 0.035
planet_sigma = 0.014
planet_signal = gaussian2d(xx, yy, planet_x, planet_y, planet_amp, planet_sigma)

# Residual stellar halo after coronagraph / PSF subtraction
# Symmetric, smooth, no spiral structure.
stellar_halo = 0.070 * np.exp(-rr / 0.33)

# Weak Airy-like residual rings around masked star
airy_like = (
    0.018 * (np.sinc(10.5 * rr) ** 2)
    + 0.010 * (np.sinc(18.0 * rr) ** 2)
)

airy_like *= np.exp(-rr / 0.80)

# Mild diffraction spikes from telescope support structure
spike_h = 0.012 * np.exp(-(yy / 0.012) ** 2) * np.exp(-np.abs(xx) / 0.65)
spike_v = 0.010 * np.exp(-(xx / 0.012) ** 2) * np.exp(-np.abs(yy) / 0.65)
diffraction_spikes = spike_h + spike_v

# Random quasi-static speckles: localized blobs, not spiral arms.
N_SPECKLES = 55
speckle_params = []

for _ in range(N_SPECKLES):
    # avoid inside coronagraph disk
    r = rng.uniform(coronagraph_radius * 1.15, 0.95)
    th = rng.uniform(-np.pi, np.pi)

    sx = r * np.cos(th)
    sy = r * np.sin(th)

    amp = rng.uniform(0.006, 0.026) * np.exp(-r / 0.70)
    sigma = rng.uniform(0.010, 0.030)

    phase = rng.uniform(0, 2 * np.pi)
    drift_phase = rng.uniform(0, 2 * np.pi)

    speckle_params.append((sx, sy, amp, sigma, phase, drift_phase))

def make_speckles(t):
    field = np.zeros_like(xx)

    for sx, sy, amp, sigma, phase, drift_phase in speckle_params:
        # tiny quasi-static drift, not large motion
        dx = 0.006 * np.sin(2 * np.pi * 0.45 * t + drift_phase)
        dy = 0.006 * np.cos(2 * np.pi * 0.38 * t + drift_phase)

        brightness = amp * (0.75 + 0.25 * np.sin(2 * np.pi * 0.35 * t + phase))

        field += gaussian2d(
            xx, yy,
            sx + dx,
            sy + dy,
            brightness,
            sigma
        )

    return field

# Coronagraph mask
mask = rr < coronagraph_radius

# Noise model:
# detector/read noise is spatially uniform over the matrix
detector_noise_sigma = 0.016

# photon noise scales mildly with local stellar halo brightness
photon_noise_scale = 0.010

static_background = stellar_halo + airy_like + diffraction_spikes

# =========================
# Figure setup
# =========================

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
ax.set_position([0, 0, 1, 1])

ax.set_xlim(-1, 1)
ax.set_ylim(-0.5625, 0.5625)
ax.set_aspect("equal")
ax.axis("off")

# =========================
# Initial image
# =========================

initial = np.zeros((N, N))

im = ax.imshow(
    initial,
    extent=[-1, 1, -1, 1],
    origin="lower",
    cmap="inferno",
    vmin=0,
    vmax=0.16,
    interpolation="nearest",
    zorder=1
)

# Coronagraph disk
corona_shadow = Circle(
    (0, 0),
    coronagraph_radius * 1.06,
    facecolor="#00030a",
    edgecolor="#9aa7b8",
    linewidth=1.2,
    alpha=0.98,
    zorder=10
)

corona_inner = Circle(
    (0, 0),
    coronagraph_radius * 0.72,
    facecolor="#020510",
    edgecolor="#1f2c3d",
    linewidth=0.8,
    alpha=1.0,
    zorder=11
)

ax.add_patch(corona_shadow)
ax.add_patch(corona_inner)

# Subtle diffraction cross
cross1, = ax.plot(
    [-0.62, 0.62],
    [0, 0],
    color=WHITE,
    linewidth=0.7,
    alpha=0.16,
    zorder=9
)

cross2, = ax.plot(
    [0, 0],
    [-0.52, 0.52],
    color=WHITE,
    linewidth=0.7,
    alpha=0.14,
    zorder=9
)

# Planet marker / HUD
planet_ring1 = Circle(
    (planet_x, planet_y),
    0.052,
    facecolor="none",
    edgecolor=PLANET,
    linewidth=1.5,
    alpha=0.0,
    zorder=20
)

planet_ring2 = Circle(
    (planet_x, planet_y),
    0.085,
    facecolor="none",
    edgecolor=PLANET,
    linewidth=0.9,
    alpha=0.0,
    zorder=20
)

planet_dot = Circle(
    (planet_x, planet_y),
    0.010,
    facecolor=PLANET,
    edgecolor=WHITE,
    linewidth=0.5,
    alpha=0.0,
    zorder=21
)

ax.add_patch(planet_ring1)
ax.add_patch(planet_ring2)
ax.add_patch(planet_dot)

connector, = ax.plot(
    [planet_x + 0.07, planet_x + 0.25],
    [planet_y + 0.04, planet_y + 0.16],
    color=PLANET,
    linewidth=0.8,
    alpha=0.0,
    zorder=22
)

label = ax.text(
    planet_x + 0.27,
    planet_y + 0.17,
    "planet candidate\nfixed detector position",
    color=PLANET,
    fontsize=10,
    fontweight="bold",
    ha="left",
    va="center",
    alpha=0.0,
    zorder=23
)

title = ax.text(
    -0.94,
    0.49,
    "CORONAGRAPHIC RESIDUAL IMAGE",
    color=TEXT,
    fontsize=18,
    fontweight="bold",
    ha="left",
    va="center",
    zorder=30
)

subtitle = ax.text(
    -0.94,
    0.445,
    "uniform detector noise + residual stellar halo + weak planet signal",
    color=MUTED,
    fontsize=11,
    ha="left",
    va="center",
    zorder=30
)

status = ax.text(
    0.94,
    -0.50,
    "planet marker: hidden",
    color=MUTED,
    fontsize=11,
    ha="right",
    va="center",
    bbox=dict(
        boxstyle="round,pad=0.35",
        facecolor="#07111f",
        edgecolor="#2b4358",
        alpha=0.85
    ),
    zorder=30
)

info = ax.text(
    -0.94,
    -0.50,
    "schematic 2D detector frame — not observed data",
    color="#415064",
    fontsize=9,
    ha="left",
    va="center",
    zorder=30
)

# =========================
# Animation helpers
# =========================

def smoothstep(edge0, edge1, value):
    t = np.clip((value - edge0) / (edge1 - edge0), 0, 1)
    return t * t * (3 - 2 * t)

def layer_alpha(t):
    fade_in = smoothstep(0.22, 0.42, t)
    fade_out = 1.0 - smoothstep(0.78, 0.94, t)
    return fade_in * fade_out

def make_frame(t):
    speckles = make_speckles(t)

    # Uniform detector/read noise across the full matrix
    detector_noise = rng.normal(
        0,
        detector_noise_sigma,
        size=(N, N)
    )

    # Photon noise is slightly stronger near residual stellar halo
    photon_noise_sigma_map = photon_noise_scale * np.sqrt(
        np.clip(static_background + speckles, 0, None)
    )

    photon_noise = rng.normal(
        0,
        photon_noise_sigma_map
    )

    frame = (
        static_background
        + speckles
        + planet_signal
        + detector_noise
        + photon_noise
    )

    # Coronagraph mask: central star blocked
    frame[mask] = rng.normal(
        0.002,
        0.002,
        size=frame[mask].shape
    )

    frame = np.clip(frame, 0, 0.18)

    return frame

# =========================
# Animation update
# =========================

def update(frame_idx):
    t = frame_idx / (FRAMES - 1)

    frame = make_frame(t)
    im.set_data(frame)

    a = layer_alpha(t)
    pulse = 0.5 + 0.5 * np.sin(2 * np.pi * 6 * t)

    planet_ring1.set_alpha((0.42 + 0.34 * pulse) * a)
    planet_ring2.set_alpha((0.22 + 0.24 * (1 - pulse)) * a)
    planet_dot.set_alpha((0.40 + 0.40 * pulse) * a)

    planet_ring1.set_radius(0.048 + 0.012 * pulse)
    planet_ring2.set_radius(0.078 + 0.024 * pulse)

    connector.set_alpha(0.72 * a)
    label.set_alpha(0.95 * a)

    if a > 0.08:
        status.set_text("planet marker: expected position")
        status.set_color(PLANET)
    else:
        status.set_text("planet marker: hidden")
        status.set_color(MUTED)

    return (
        im,
        planet_ring1,
        planet_ring2,
        planet_dot,
        connector,
        label,
        status,
    )

# =========================
# Render and export
# =========================

frames = []
for frame_idx in range(FRAMES):
    update(frame_idx)
    fig.canvas.draw()
    rgba = np.asarray(fig.canvas.buffer_rgba(), dtype=np.uint8)
    frames.append(rgba[..., :3].copy())

plt.close(fig)

out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
)

if OUTPUT_FORMAT == "gif":
    display(Image(filename=str(out_file)))
else:
    display(Video(filename=str(out_file), embed=False))

print(f"Saved: {out_file}")
